<a href="https://colab.research.google.com/github/ihattab/ContextCraft/blob/main/KubeOrchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
KubeOrchestrator × Antigravity
================================
A code sample demonstrating the multi-agent orchestration architecture
for autonomous Kubernetes operations, powered by Antigravity primitives.

Architecture layers demonstrated:
  1. /goal entry point          → Fully Autonomous Goal Execution
  2. Root orchestrator          → Dynamic Subagents & Shared Agent Harness
  3. Isolated subagents         → Isolated Git Worktree Mode (simulated)
  4. Parallel execution         → Non-Blocking Async Task Queues
  5. Shared harness             → Output merge + conflict resolution
  6. Safety gate                → Declarative Safety Policies & Lifecycle Hooks
  7. Token budget monitor       → Zero-Overhead Token Budget & Accumulation Monitor
  8. Human-in-the-loop          → Interactive Approval Gatel
  9. Execution                  → kubectl apply (approved patches only)
 10. Scheduled sidecar          → Scheduled Tasks & Uncoupled Projects
 11. Browser verification       → Continuous Verification & Browser Actuation
 12. SKILL.md engines           → Serverless Ad-hoc Skill Execution Engines
"""

import asyncio
import json
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVES: Token Budget Monitor (SDK layer)
# ─────────────────────────────────────────────

class TokenBudgetExceeded(Exception):
    pass

class TokenBudgetMonitor:
    """
    Zero-Overhead Token Budget & Accumulation Monitor.
    Tracks per-turn and cumulative token usage.
    Triggers a kill-switch if the budget is exceeded.
    """
    def __init__(self, max_tokens: int = 50_000):
        self.max_tokens = max_tokens
        self.used = 0
        self.per_turn: list[int] = []

    def consume(self, tokens: int, label: str = ""):
        self.used += tokens
        self.per_turn.append(tokens)
        print(f"  [TokenMonitor] {label}: +{tokens} tokens  |  total: {self.used}/{self.max_tokens}")
        if self.used > self.max_tokens:
            raise TokenBudgetExceeded(
                f"Kill-switch triggered: {self.used} tokens exceeds budget of {self.max_tokens}"
            )

    def report(self):
        return {"total_used": self.used, "budget": self.max_tokens, "turns": len(self.per_turn)}

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVES: Declarative Safety Policies (SDK layer)
# ─────────────────────────────────────────────

class Decision(Enum):
    ALLOW = "allow"
    DENY  = "deny"
    TRANSFORM = "transform"

@dataclass
class SafetyPolicy:
    """
    Declarative Safety Policy — deny(*) by default.
    Inspect → Decide → Transform lifecycle hooks.
    """
    name: str
    inspect:   Callable[[dict], bool]   # returns True if this policy applies
    decide:    Callable[[dict], Decision]
    transform: Callable[[dict], dict] | None = None

class SafetyGate:
    """
    Human-in-the-loop approval gate.
    All policies are deny-by-default; each command
    must pass every registered policy to be executed.
    """
    def __init__(self, token_monitor: TokenBudgetMonitor, human_approval: bool = True):
        self.policies: list[SafetyPolicy] = []
        self.token_monitor = token_monitor
        self.human_approval = human_approval
        self._register_defaults()

    def _register_defaults(self):
        # Deny all destructive kubectl operations by default
        self.register(SafetyPolicy(
            name="deny-destructive-kubectl",
            inspect=lambda cmd: cmd.get("tool") == "kubectl" and
                               any(op in cmd.get("args", "") for op in ["delete", "drain", "cordon"]),
            decide=lambda cmd: Decision.DENY,
        ))
        # Require approval for any kubectl apply
        self.register(SafetyPolicy(
            name="require-approval-for-apply",
            inspect=lambda cmd: cmd.get("tool") == "kubectl" and "apply" in cmd.get("args", ""),
            decide=lambda cmd: Decision.ALLOW,   # allowed only after human gate below
        ))
        # Transform: strip --force flags automatically
        self.register(SafetyPolicy(
            name="strip-force-flag",
            inspect=lambda cmd: "--force" in cmd.get("args", ""),
            decide=lambda cmd: Decision.TRANSFORM,
            transform=lambda cmd: {**cmd, "args": cmd["args"].replace("--force", "").strip()},
        ))

    def register(self, policy: SafetyPolicy):
        self.policies.append(policy)

    def evaluate(self, command: dict) -> tuple[Decision, dict]:
        """
        Runs Inspect → Decide → Transform for each matching policy.
        First DENY wins. TRANSFORM mutates the command in place.
        """
        self.token_monitor.consume(50, f"SafetyGate.evaluate({command.get('tool')})")
        for policy in self.policies:
            if policy.inspect(command):
                decision = policy.decide(command)
                print(f"  [SafetyGate] Policy '{policy.name}' → {decision.value}")
                if decision == Decision.DENY:
                    return Decision.DENY, command
                if decision == Decision.TRANSFORM and policy.transform:
                    command = policy.transform(command)
                    print(f"  [SafetyGate] Transformed command: {command}")
        return Decision.ALLOW, command

    def human_gate(self, command: dict) -> bool:
        """
        Interactive Human-in-the-Loop Approval Gate.
        In production this would pause and poll for manual confirmation.
        """
        if not self.human_approval:
            return True
        print(f"\n  [HumanGate] Pending approval for: kubectl {command['args']}")
        print("  [HumanGate] Simulating operator approval... ✓")
        return True

In [ ]:
# ─────────────────────────────────────────────
# PRIMITIVES: Git Worktree (Isolated subagent context)
# ─────────────────────────────────────────────

@dataclass
class GitWorktree:
    """
    Isolated Git Worktree Mode.
    Each subagent operates in its own sandboxed branch.
    No agent ever mutates live manifests directly.
    """
    agent_name: str
    branch: str = field(init=False)
    patches: list[dict] = field(default_factory=list)

    def __post_init__(self):
        self.branch = f"agent/{self.agent_name.lower().replace(' ', '-')}/{int(time.time())}"
        print(f"  [Worktree] Created isolated branch: {self.branch}")

    def stage_patch(self, manifest: str, change: dict):
        patch = {"manifest": manifest, "change": change, "branch": self.branch}
        self.patches.append(patch)
        print(f"  [Worktree] Staged patch to {manifest} in {self.branch}")
        return patch

    def diff(self) -> list[dict]:
        return self.patches

In [ ]:
# ─────────────────────────────────────────────
# SKILL.md ENGINES: Ad-hoc tool skill packs
# ─────────────────────────────────────────────

class SkillEngine:
    """
    Serverless Ad-hoc Skill Execution Engine.
    Loads ephemeral SKILL.md instruction packs per task.
    No persistent server runtime required.
    """
    SKILLS = {
        "kubectl": {"version": "1.29", "verbs": ["get", "apply", "describe", "top"]},
        "helm":    {"version": "3.14", "verbs": ["template", "diff", "upgrade"]},
        "opa":     {"version": "0.63", "verbs": ["eval", "check"]},
    }

    @classmethod
    def load(cls, skill: str) -> dict:
        if skill not in cls.SKILLS:
            raise ValueError(f"No SKILL.md found for '{skill}'")
        print(f"  [SkillEngine] Loaded ephemeral skill pack: {skill} v{cls.SKILLS[skill]['version']}")
        return cls.SKILLS[skill]

    @classmethod
    def run(cls, skill: str, verb: str, args: str) -> dict:
        spec = cls.load(skill)
        if verb not in spec["verbs"]:
            raise ValueError(f"Verb '{verb}' not in skill '{skill}'")
        print(f"  [SkillEngine] Executing: {skill} {verb} {args}")
        return {"tool": skill, "verb": verb, "args": f"{verb} {args}", "status": "ok"}

In [ ]:
# ─────────────────────────────────────────────
# SUBAGENTS: Four specialized agents
# ─────────────────────────────────────────────

@dataclass
class SubagentResult:
    agent: str
    findings: dict
    patches: list[dict]
    tokens_used: int

async def metrics_agent(namespace: str, monitor: TokenBudgetMonitor) -> SubagentResult:
    """
    Metrics subagent — reads Prometheus/Grafana data.
    Runs in isolated git worktree context.
    """
    print("\n  [MetricsAgent] Starting in isolated worktree...")
    worktree = GitWorktree("metrics-agent")
    await asyncio.sleep(0.1)  # simulate async I/O

    monitor.consume(800, "MetricsAgent.query_prometheus")

    # Simulated Prometheus metrics
    findings = {
        "cpu_request_waste_pct": 67,
        "memory_request_waste_pct": 43,
        "top_wasteful_pods": ["api-server-7f9b", "worker-3c2a", "cache-1d8e"],
    }
    print(f"  [MetricsAgent] CPU waste: {findings['cpu_request_waste_pct']}%  |  "
          f"Memory waste: {findings['memory_request_waste_pct']}%")
    return SubagentResult(agent="metrics", findings=findings, patches=[], tokens_used=800)


async def remediation_agent(findings: dict, namespace: str, monitor: TokenBudgetMonitor) -> SubagentResult:
    """
    Remediation subagent — proposes kubectl patches.
    Stages all changes in its own isolated worktree branch.
    """
    print("\n  [RemediationAgent] Starting in isolated worktree...")
    worktree = GitWorktree("remediation-agent")
    await asyncio.sleep(0.1)

    monitor.consume(1200, "RemediationAgent.generate_patches")

    patches = []
    for pod in findings.get("top_wasteful_pods", []):
        patch = worktree.stage_patch(
            manifest=f"{namespace}/{pod}.yaml",
            change={"resources": {"requests": {"cpu": "100m", "memory": "128Mi"}}}
        )
        patches.append(patch)

    print(f"  [RemediationAgent] Staged {len(patches)} resource patches")
    return SubagentResult(agent="remediation", findings={}, patches=patches, tokens_used=1200)


async def cost_agent(namespace: str, monitor: TokenBudgetMonitor) -> SubagentResult:
    """
    Cost subagent — analyses resource requests vs limits.
    """
    print("\n  [CostAgent] Starting in isolated worktree...")
    worktree = GitWorktree("cost-agent")
    await asyncio.sleep(0.1)

    monitor.consume(600, "CostAgent.analyze_costs")

    findings = {
        "monthly_waste_usd": 340,
        "projected_savings_usd": 228,
        "savings_pct": 67,
        "over_provisioned_deployments": 8,
    }
    print(f"  [CostAgent] Projected savings: ${findings['projected_savings_usd']}/mo "
          f"({findings['savings_pct']}% reduction)")
    return SubagentResult(agent="cost", findings=findings, patches=[], tokens_used=600)


async def compliance_agent(namespace: str, monitor: TokenBudgetMonitor) -> SubagentResult:
    """
    Compliance subagent — validates patches against OPA policy baselines.
    """
    print("\n  [ComplianceAgent] Starting in isolated worktree...")
    worktree = GitWorktree("compliance-agent")
    await asyncio.sleep(0.1)

    monitor.consume(700, "ComplianceAgent.validate_policies")

    SkillEngine.load("opa")
    findings = {
        "policy_violations": 0,
        "warnings": ["memory limits not set on 2 pods"],
        "compliant": True,
    }
    print(f"  [ComplianceAgent] Policy violations: {findings['policy_violations']}  |  "
          f"Warnings: {len(findings['warnings'])}")
    return SubagentResult(agent="compliance", findings=findings, patches=[], tokens_used=700)

In [ ]:
# ─────────────────────────────────────────────
# SHARED AGENT HARNESS
# ─────────────────────────────────────────────

class SharedAgentHarness:
    """
    Shared Agent Harness — merges subagent outputs,
    resolves conflicts, and surfaces a unified diff
    to the operator before any live cluster mutation.

    This is the core of the Dynamic Subagents architecture:
    each subagent runs in isolation, the harness coordinates.
    """
    def __init__(self, monitor: TokenBudgetMonitor):
        self.monitor = monitor

    def merge(self, results: list[SubagentResult]) -> dict:
        print("\n  [Harness] Merging subagent outputs...")
        self.monitor.consume(500, "Harness.merge")

        all_patches = []
        all_findings = {}
        for r in results:
            all_patches.extend(r.patches)
            all_findings[r.agent] = r.findings

        # Conflict resolution: deduplicate patches by manifest
        seen = set()
        deduped = []
        for p in all_patches:
            key = p["manifest"]
            if key not in seen:
                seen.add(key)
                deduped.append(p)
            else:
                print(f"  [Harness] Conflict resolved: duplicate patch for {key} dropped")

        unified_diff = {
            "namespace":  all_findings.get("metrics", {}).get("top_wasteful_pods", []),
            "patches":    deduped,
            "cost":       all_findings.get("cost", {}),
            "compliance": all_findings.get("compliance", {}),
            "patch_count": len(deduped),
        }
        print(f"  [Harness] Unified diff ready: {len(deduped)} patches across "
              f"{len(results)} subagent workstreams")
        return unified_diff

In [ ]:
# ─────────────────────────────────────────────
# SCHEDULED SIDECAR
# ─────────────────────────────────────────────

class ScheduledSidecar:
    """
    Scheduled Tasks & Uncoupled Projects.
    Runs routine operations on a cron schedule
    without a GUI open — true headless operation.
    """
    def __init__(self, interval_seconds: int = 300):
        self.interval = interval_seconds
        self.job_count = 0

    async def run_once(self, namespace: str):
        self.job_count += 1
        print(f"\n  [Sidecar] Scheduled health check #{self.job_count} for namespace: {namespace}")
        await asyncio.sleep(0.05)
        print("  [Sidecar] Health check complete — no critical issues detected")

    async def start(self, namespace: str, runs: int = 1):
        """In production this loops indefinitely on cron schedule."""
        for _ in range(runs):
            await self.run_once(namespace)


# ─────────────────────────────────────────────
# BROWSER ACTUATION
# ─────────────────────────────────────────────

class BrowserActuator:
    """
    Continuous Verification & Browser Actuation.
    Uses headless Chrome to verify post-deploy UI state.
    Maps to Antigravity's /browser command layer.
    """
    async def verify(self, service_url: str) -> dict:
        print(f"\n  [BrowserActuator] /browser verify {service_url}")
        await asyncio.sleep(0.1)  # simulate headless Chrome
        result = {
            "url": service_url,
            "status": "200 OK",
            "load_time_ms": 312,
            "visual_regression": False,
            "assertions_passed": 7,
        }
        print(f"  [BrowserActuator] ✓ {result['status']}  |  "
              f"{result['assertions_passed']} assertions passed  |  "
              f"Visual regression: {result['visual_regression']}")
        return result

In [ ]:
# ─────────────────────────────────────────────
# ROOT ORCHESTRATOR
# ─────────────────────────────────────────────

class RootOrchestrator:
    """
    Root Orchestrator Agent.
    Receives the /goal command, decomposes it,
    spawns Dynamic Subagents in parallel via async task queues,
    and coordinates through the Shared Agent Harness.

    This is the core Antigravity pattern:
    tree-based orchestration to avoid flat-prompt context saturation.
    """
    def __init__(self):
        self.monitor  = TokenBudgetMonitor(max_tokens=20_000)
        self.harness  = SharedAgentHarness(self.monitor)
        self.gate     = SafetyGate(self.monitor, human_approval=True)
        self.sidecar  = ScheduledSidecar(interval_seconds=300)
        self.browser  = BrowserActuator()

    async def execute_goal(self, goal: str, namespace: str):
        """
        Fully Autonomous Goal Execution — /goal entry point.
        Runs iteratively until the complex task is verified complete.
        """
        print(f"\n{'═'*60}")
        print(f"  /goal  {goal}")
        print(f"  namespace: {namespace}")
        print(f"{'═'*60}")

        self.monitor.consume(200, "Orchestrator.parse_goal")

        # ── STEP 1: Spawn subagents in parallel (Non-Blocking Async Task Queues)
        print("\n[ Step 1 ] Spawning isolated subagents in parallel...")
        results: list[SubagentResult] = await asyncio.gather(
            metrics_agent(namespace, self.monitor),
            remediation_agent(
                {"top_wasteful_pods": ["api-server-7f9b", "worker-3c2a", "cache-1d8e"]},
                namespace,
                self.monitor,
            ),
            cost_agent(namespace, self.monitor),
            compliance_agent(namespace, self.monitor),
        )
        print(f"\n  All {len(results)} subagents completed.")

        # ── STEP 2: Shared Agent Harness — merge + resolve conflicts
        print("\n[ Step 2 ] Shared Agent Harness: merging outputs...")
        unified_diff = self.harness.merge(results)

        # ── STEP 3: Safety Gate — Inspect → Decide → Transform
        print("\n[ Step 3 ] Safety Gate evaluation...")
        kubectl_cmd = SkillEngine.run(
            "kubectl", "apply",
            f"-f patches/ -n {namespace} --dry-run=client"
        )
        decision, final_cmd = self.gate.evaluate(kubectl_cmd)

        if decision == Decision.DENY:
            print("  [SafetyGate] ✗ Command denied. Halting execution.")
            return

        # ── STEP 4: Human-in-the-loop approval
        print("\n[ Step 4 ] Human-in-the-loop approval gate...")
        approved = self.gate.human_gate(final_cmd)
        if not approved:
            print("  [HumanGate] ✗ Operator rejected. No changes applied.")
            return

        # ── STEP 5: Apply approved patches only
        print("\n[ Step 5 ] Applying approved patches...")
        print(f"  kubectl {final_cmd['args']}")
        print(f"  → Applied {unified_diff['patch_count']} patches to namespace '{namespace}'")
        projected = unified_diff["cost"].get("projected_savings_usd", 0)
        print(f"  → Projected savings: ${projected}/mo")

        # ── STEP 6: Browser verification
        print("\n[ Step 6 ] Post-deploy browser verification...")
        await self.browser.verify(f"https://{namespace}.internal/health")

        # ── STEP 7: Scheduled sidecar
        print("\n[ Step 7 ] Scheduled sidecar: post-apply health check...")
        await self.sidecar.start(namespace, runs=1)

        # ── Final report
        print(f"\n{'═'*60}")
        print("  /goal  COMPLETE")
        budget = self.monitor.report()
        print(f"  Tokens used: {budget['total_used']} / {budget['budget']}  "
              f"({budget['turns']} turns)")
        print(f"{'═'*60}\n")

In [ ]:
# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

async def main():
    orchestrator = RootOrchestrator()
    await orchestrator.execute_goal(
        goal="Optimise namespace production for cost by 30%",
        namespace="production",
    )

# Colab automatically manages the async event loop via nest_asyncio in the background.
# Since an event loop is already running in Colab, await the main function directly.
await main()


════════════════════════════════════════════════════════════
  /goal  Optimise namespace production for cost by 30%
  namespace: production
════════════════════════════════════════════════════════════
  [TokenMonitor] Orchestrator.parse_goal: +200 tokens  |  total: 200/20000

[ Step 1 ] Spawning isolated subagents in parallel...

  [MetricsAgent] Starting in isolated worktree...
  [Worktree] Created isolated branch: agent/metrics-agent/1781140582

  [RemediationAgent] Starting in isolated worktree...
  [Worktree] Created isolated branch: agent/remediation-agent/1781140582

  [CostAgent] Starting in isolated worktree...
  [Worktree] Created isolated branch: agent/cost-agent/1781140582

  [ComplianceAgent] Starting in isolated worktree...
  [Worktree] Created isolated branch: agent/compliance-agent/1781140582
  [TokenMonitor] MetricsAgent.query_prometheus: +800 tokens  |  total: 1000/20000
  [MetricsAgent] CPU waste: 67%  |  Memory waste: 43%
  [TokenMonitor] RemediationAgent.generate_p